In [ ]:
import numpy as np

class DotsAndBoxesStates:
    def __init__(self, h, w):
        self.h = h
        self.w = w
        self.horizontal_edges = np.zeros((h, w-1), dtype=int)
        self.vertical_edges = np.zeros((h-1, w), dtype=int)
        self.player_turn = 1
        self.scores = {1: 0, 2: 0}
        self.boxes = np.zeros((h-1, w-1), dtype=int)
    
    def apply_move(self, move_type, r, c):
        if move_type == 'h':
            self.horizontal_edges[r, c] = 1
        else:
            self.vertical_edges[r, c] = 1

        boxes_completed = self._check_and_update_boxes()
        if boxes_completed > 0:
            self.scores[self.player_turn] += boxes_completed
        else:
            self.player_turn = 2 if self.player_turn == 1 else 1
        return self

    def _check_and_update_boxes(self):
        new_boxes = 0
        for r in range(self.h - 1):
            for c in range(self.w - 1):
                if self.boxes[r, c] == 0:
                    top = self.horizontal_edges[r, c]
                    bottom = self.horizontal_edges[r + 1, c]
                    left = self.vertical_edges[r, c]
                    right = self.vertical_edges[r, c + 1]
                    if top and bottom and left and right:
                        self.boxes[r, c] = self.player_turn
                        new_boxes += 1
        
        return new_boxes

    def get_legal_moves(self):
        moves = []
        for r in range(self.h):
            for c in range(self.w - 1):
                if self.horizontal_edges[r, c] == 0: moves.append(('h', r, c))
        for r in range(self.h - 1):
            for c in range (self.w):
                if self.vertical_edges[r, c] == 0: moves.append(('v', r, c))
        return moves
    
    def is_terminal(self):
        return len(self.get_legal_moves()) == 0

    def display(self):
        print(f"Score: P1: {self.scores[1]} | P2: {self.scores[2]}")
        print(f"Turn: Player {self.player_turn}")
        for r in range(self.h):
            line = ""
            for c in range(self.w - 1):
                line += "● " + ("— " if self.horizontal_edges[r, c] == 1 else "  ")
            line += "●"
            print(line)
            if r < self.h - 1:
                v_line = ""
                for c in range(self.w):
                    v_line += "| " if self.vertical_edges[r, c] == 1 else "  "
                    if c < self.w - 1:
                        v_line += f"{self.boxes[r, c] if self.boxes[r, c] != 0 else ' '} "
                print(v_line)
    
    def clone(self):
        new_state = DotsAndBoxesStates(self.h, self.w)
        new_state.horizontal_edges = self.horizontal_edges.copy()
        new_state.vertical_edges = self.vertical_edges.copy()
        new_state.boxes = self.boxes.copy()
        new_state.player_turn = self.player_turn
        new_state.scores = self.scores.copy()
        return new_state
    

In [59]:
# Example state and moves
state = DotsAndBoxesStates(3, 3)
turn = state.player_turn

print(f"Current turn: Player {turn}")
state.apply_move('h', 0, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('v', 0, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('h', 1, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('v', 0, 1)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.display()

Current turn: Player 1
Current turn: Player 2
Current turn: Player 1
Current turn: Player 2
Current turn: Player 2
Score: P1: 0 | P2: 1
Turn: Player 2
● — ●   ●
| 2 |     
● — ●   ●
          
●   ●   ●


In [60]:
import math
import random

class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children = []
        self.visits = 0
        self.wins = 0.0
        self.untried_actions = state.get_legal_moves()

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, c_param=1.4):
        # UCB1 = (wins / visits) + C * sqrt(log(parent_visits) / visits)
        choices_weights = [
            (child.wins / child.visits) + c_param * math.sqrt((2 * math.log(self.visits) / child.visits))
            for child in self.children
        ]
        return self.children[choices_weights.index(max(choices_weights))]

    def expand(self):
        move = self.untried_actions.pop()
        next_state = self.state.clone()
        next_state.apply_move(*move)
        child_node = MCTSNode(next_state, parent=self, move=move)
        self.children.append(child_node)
        return child_node

In [61]:
# Simulate a random playout from the given node's state

root = MCTSNode(DotsAndBoxesStates(3, 3))
root.expand()
root.expand()
root.expand()

print ("After expanding root node 3 times")
print(f"Check Children: {len(root.children)}")
print (f"Untried Actions: {len(root.untried_actions)}\n\n")

print(f"Root state:")
root.state.display()
print("\n\n")

print(f"First child state:")
root.children[0].state.display()

After expanding root node 3 times
Check Children: 3
Untried Actions: 9


Root state:
Score: P1: 0 | P2: 0
Turn: Player 1
●   ●   ●
          
●   ●   ●
          
●   ●   ●



First child state:
Score: P1: 0 | P2: 0
Turn: Player 2
●   ●   ●
          
●   ●   ●
        | 
●   ●   ●


In [62]:
class MCTS:
    def __init__(self, iterations=1000):
        self.iterations = iterations
    
    def search(self, initial_state):
        self.root = MCTSNode(state=initial_state.clone())

        for _ in range(self.iterations):
            # 1. Selection
            node = self.select(self.root)
            # 2. Expansion
            if not node.state.is_terminal():
                node = node.expand()
            # 3. Simulation
            result = self.simulate(node.state)
            # 4. Backpropagation
            self.backpropagate(node, result)

        return max(self.root.children, key=lambda n: n.visits).move

    def select(self, node):
        while node.is_fully_expanded() and not node.state.is_terminal():
            node = node.best_child()
        return node

    def simulate(self, state):
        temp_state = state.clone()
        while not temp_state.is_terminal():
            moves = temp_state.get_legal_moves()
            
            # Winning Move
            win_moves = [m for m in moves if self._would_complete(temp_state, m)]
            if win_moves:
                move = random.choice(win_moves)
            else:
                # Safe move or Random move
                safe_moves = [m for m in moves if temp_state._is_safe_move(m)]
                move = random.choice(safe_moves) if safe_moves else random.choice(moves)
            
            temp_state.apply_move(*move)
        
        # Normalized
        total = (temp_state.h - 1) * (temp_state.w - 1)
        return (temp_state.scores[1] - temp_state.scores[2]) / total

    def _would_complete(self, state, move):
        m_type, r, c = move
        if m_type == 'h':
            if r > 0 and state._count_sides(r-1, c) == 3: return True
            if r < state.h - 1 and state._count_sides(r, c) == 3: return True
        else:
            if c > 0 and state._count_sides(r, c-1) == 3: return True
            if c < state.w - 1 and state._count_sides(r, c) == 3: return True
        return False
    
    def backpropagate(self, node, result):
        while node is not None:
            node.visits += 1
            if node.parent:
                # Reward based on whose turn it was to move to this node
                mover = node.parent.state.player_turn
                node.wins += result if mover == 1 else -result
            node = node.parent

In [63]:
import time
from IPython.display import clear_output

# Main game loop
board_size = 5
game = DotsAndBoxesStates(board_size, board_size)
ai_p1 = MCTS(iterations=600)
ai_p2 = MCTS(iterations=600)

while not game.is_terminal():
    clear_output(wait=True)
    game.display()
    
    current_ai = ai_p1 if game.player_turn == 1 else ai_p2
    print(f"\nAI Player {game.player_turn} is thinking...")
    
    move = current_ai.search(game)
    game.apply_move(*move)
    
    time.sleep(0.1) # Delay for visualization

clear_output(wait=True)
game.display()
print("\nGAME OVER!")
if game.scores[1] > game.scores[2]: print("Winner: Player 1")
elif game.scores[2] > game.scores[1]: print("Winner: Player 2")
else: print("It's a Tie!")

print(f"Final Scores: Player 1: {game.scores[1]}, Player 2: {game.scores[2]}")

Score: P1: 2 | P2: 14
Turn: Player 2
● — ● — ● — ● — ●
| 2 | 2 | 1 | 1 | 
● — ● — ● — ● — ●
| 2 | 2 | 2 | 2 | 
● — ● — ● — ● — ●
| 2 | 2 | 2 | 2 | 
● — ● — ● — ● — ●
| 2 | 2 | 2 | 2 | 
● — ● — ● — ● — ●

GAME OVER!
Winner: Player 2
Final Scores: Player 1: 2, Player 2: 14
